# Self-Consistency Chain-of-Thought Agent

## Educational Demo for Data Science Students

This notebook demonstrates the **Self-Consistency** approach for improving Large Language Model (LLM) reasoning accuracy through multiple sampling and majority voting.

### Key Concepts:
- **Chain-of-Thought (CoT)**: LLMs provide step-by-step reasoning
- **Self-Consistency**: Generate multiple reasoning paths and select the most frequent answer
- **Mathematical Foundation**: `argmax_a Σ_{i=1}^m 𝟙_a(a_i = a)` (majority vote)
- **Algorithmic Optimization**: O(m) complexity using Python's `Counter`

### Prerequisites:
1. **LiteLLM Server**: Run `make litellm-install` to start local LLM proxy
2. **Environment**: Configure `.env` file with your LLM settings
3. **Dependencies**: Install with `make install`


## 1. Setup and Imports

First, let's import our self-consistency agent components and set up the environment.

In [ ]:
# Core imports for the self-consistency agent
import os
import sys
from pathlib import Path

# Add the project root to Python path so we can import our modules
project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Import our self-consistency agent components
from llm_agents.self_consistency.domain import LLMResponse, ConsensusResult
from llm_agents.self_consistency.interfaces import LiteLLMAdapter
from llm_agents.self_consistency.config import AgentConfig
from llm_agents.self_consistency.agent import SelfConsistencyAgent

# Data science libraries for analysis and visualization
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import time
import numpy as np

# Set up plotting style
plt.style.use('default')
sns.set_palette("husl")

print("✅ All imports successful!")
print(f"📁 Working directory: {project_root}")

## 2. Environment Configuration Check

Let's verify our environment is properly configured for LLM interaction.

In [ ]:
%load_ext dotenv
%dotenv

# Check environment configuration
def check_environment():
    """Check if environment is properly configured."""
    config_status = {
        'LLM_MODEL': os.getenv('LLM_MODEL', 'claude-3-haiku'),  # Default from .env.example
        'LLM_BASE_URL': os.getenv('LLM_BASE_URL', 'http://localhost:4000'),
        'LLM_TEMPERATURE': os.getenv('LLM_TEMPERATURE', '0.7'),
        'LLM_API_KEY': os.getenv('LLM_API_KEY', 'sk-1234')[:20] + '...' if os.getenv('LLM_API_KEY') else 'sk-1234'
    }
    
    print("🔧 Environment Configuration:")
    for key, value in config_status.items():
        print(f"  {key}: {value}")
    
    # Check if .env file exists
    env_file = project_root / '.env'
    if env_file.exists():
        print("\n✅ .env file found")
    else:
        print("\n⚠️  .env file not found")
        print("   Run: make setup-env to create from template")
    
    return config_status

config = check_environment()

## 3. Test LLM Connection

Before running our experiments, let's test the connection to our LiteLLM server.

In [ ]:
# Test LLM connection
def test_llm_connection():
    """Test connection to LiteLLM server."""
    try:
        print("🔌 Testing LLM connection...")
        
        # Create LLM adapter
        adapter = LiteLLMAdapter()
        print(f"📡 Connecting to: {adapter.base_url}")
        print(f"🤖 Using model: {adapter.model}")
        
        # Test simple query
        start_time = time.time()
        response = adapter.generate_llm_response(
            prompt="Think step by step and provide your reasoning.",
            question="What is 2 + 2?"
        )
        end_time = time.time()
        
        print(f"\n✅ Connection successful! ({end_time - start_time:.2f}s)")
        print(f"💭 Reasoning: {response.reasoning[:100]}...")
        print(f"🎯 Answer: {response.answer}")
        
        return adapter, True
        
    except Exception as e:
        print(f"\n❌ Connection failed: {e}")
        print("\n🔧 Troubleshooting:")
        print("   1. Check if LiteLLM is running: make litellm-status")
        print("   2. Start LiteLLM if needed: make litellm-install")
        print("   3. Test connection: make litellm-test")
        
        return None, False

adapter, connection_ok = test_llm_connection()

## 4. Basic Self-Consistency Example

Let's start with a simple example to demonstrate the self-consistency approach.

In [ ]:
# Basic self-consistency demonstration
if connection_ok:
    print("🧪 Basic Self-Consistency Demo")
    print("=" * 40)
    
    # Configure the agent
    config = AgentConfig(
        llm_interface=adapter,
        target_responses=5,  # Generate 5 responses for consensus
        prompt_template="Think step by step and provide your reasoning. End with 'The answer is [your answer]'."
    )
    
    # Test question
    question = "If a train travels 60 km in 45 minutes, what is its speed in km/h?"
    print(f"📝 Question: {question}")
    
    # Create and run the agent
    agent = SelfConsistencyAgent(config, question)
    
    print(f"\n🔄 Generating {config.target_responses} responses...")
    start_time = time.time()
    
    result = agent.process_question()
    
    end_time = time.time()
    
    # Display results
    print(f"\n⏱️  Total time: {end_time - start_time:.2f}s")
    print(f"\n🎯 Final Answer: {result.final_answer}")
    print(f"📊 Confidence: {result.confidence:.1%} ({result.vote_count}/{config.target_responses} votes)")
    
    # Show individual responses
    print(f"\n📋 Individual Responses:")
    for i, response in enumerate(agent._llm_responses, 1):
        print(f"  {i}. Answer: {response.answer}")
        print(f"     Reasoning: {response.reasoning[:80]}...")
        
else:
    print("⚠️  Skipping demo - LLM connection not available")
    print("   Please fix connection issues and re-run this cell")

## 5. Comparative Analysis: Single vs Multiple Responses

Let's compare the performance of single response vs self-consistency across multiple questions.

In [ ]:
# Comparative analysis function
def run_comparative_analysis():
    """Compare single response vs self-consistency on multiple questions."""
    
    # Test questions with known correct answers
    test_questions = [
        {
            "question": "What is 15% of 240?",
            "correct_answer": "36",
            "category": "Math"
        },
        {
            "question": "If you buy 3 items for $4.50 each, how much change do you get from $20?",
            "correct_answer": "$6.50",
            "category": "Math"
        },
        {
            "question": "A rectangle has length 8cm and width 5cm. What is its area?",
            "correct_answer": "40 cm²",
            "category": "Geometry"
        },
        {
            "question": "Convert 2.5 hours to minutes.",
            "correct_answer": "150 minutes",
            "category": "Conversion"
        }
    ]
    
    results = []
    
    for test_case in test_questions:
        question = test_case["question"]
        print(f"\n📝 Testing: {question}")
        
        # Single response approach
        print("   🔸 Single response...")
        single_response = adapter.generate_llm_response(
            "Think step by step and provide your answer.", 
            question
        )
        
        # Self-consistency approach (3 responses for speed)
        print("   🔸 Self-consistency (3 responses)...")
        sc_config = AgentConfig(llm_interface=adapter, target_responses=3)
        sc_agent = SelfConsistencyAgent(sc_config, question)
        sc_result = sc_agent.process_question()
        
        # Collect results
        results.append({
            'question': question,
            'category': test_case['category'],
            'correct_answer': test_case['correct_answer'],
            'single_response': single_response.answer,
            'sc_response': sc_result.final_answer,
            'sc_confidence': sc_result.confidence,
            'sc_votes': f"{sc_result.vote_count}/{sc_config.target_responses}"
        })
    
    return pd.DataFrame(results)

if connection_ok:
    print("🔬 Running Comparative Analysis...")
    print("This may take a few minutes...")
    
    comparison_df = run_comparative_analysis()
    
    print("\n📊 Results Summary:")
    print(comparison_df[['question', 'correct_answer', 'single_response', 'sc_response', 'sc_confidence']].to_string(index=False))
else:
    print("⚠️  Skipping analysis - LLM connection not available")

## 6. Algorithmic Complexity Analysis

Let's demonstrate the O(m) complexity of our majority voting implementation.

In [ ]:
# Algorithmic complexity demonstration
def demonstrate_complexity():
    """Demonstrate O(m) complexity of majority voting."""
    
    print("⚡ Algorithmic Complexity Analysis")
    print("=" * 40)
    
    # Create mock responses for testing
    def create_mock_responses(m):
        """Create m mock responses for performance testing."""
        responses = []
        # Create a distribution: 60% answer A, 30% answer B, 10% answer C
        for i in range(m):
            if i < m * 0.6:
                answer = "A"
            elif i < m * 0.9:
                answer = "B"
            else:
                answer = "C"
            
            responses.append(LLMResponse(
                reasoning=f"Mock reasoning {i}",
                answer=answer
            ))
        return responses
    
    # Test different values of m
    m_values = [10, 50, 100, 500, 1000, 5000]
    timing_results = []
    
    for m in m_values:
        print(f"📏 Testing with m = {m} responses...")
        
        # Create mock agent with responses
        mock_config = AgentConfig(llm_interface=None, target_responses=m)
        mock_agent = SelfConsistencyAgent(mock_config, "test question")
        mock_agent._llm_responses = create_mock_responses(m)
        
        # Time the argmax operation
        start_time = time.time()
        answer, count = mock_agent._perform_argmax()
        end_time = time.time()
        
        execution_time = (end_time - start_time) * 1000  # Convert to milliseconds
        timing_results.append({
            'm': m,
            'time_ms': execution_time,
            'answer': answer,
            'votes': count
        })
        
        print(f"   ⏱️  Time: {execution_time:.3f}ms, Winner: {answer} ({count} votes)")
    
    return pd.DataFrame(timing_results)

# Run complexity analysis
timing_df = demonstrate_complexity()

# Visualize complexity
plt.figure(figsize=(10, 6))
plt.subplot(1, 2, 1)
plt.plot(timing_df['m'], timing_df['time_ms'], 'bo-', linewidth=2, markersize=8)
plt.xlabel('Number of Responses (m)')
plt.ylabel('Execution Time (ms)')
plt.title('Majority Vote Performance\n(O(m) Linear Complexity)')
plt.grid(True, alpha=0.3)

# Show linear relationship
plt.subplot(1, 2, 2)
plt.plot(timing_df['m'], timing_df['time_ms'] / timing_df['m'], 'ro-', linewidth=2, markersize=8)
plt.xlabel('Number of Responses (m)')
plt.ylabel('Time per Response (ms)')
plt.title('Time per Response\n(Should be roughly constant)')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n📈 Performance Summary:")
print(f"   • Largest test (m=5000): {timing_df.iloc[-1]['time_ms']:.3f}ms")
print(f"   • Average time per response: {(timing_df['time_ms'] / timing_df['m']).mean():.6f}ms")
print(f"   • Complexity: O(m) - Linear scaling confirmed! ✅")

## 7. Confidence Analysis

Let's analyze how confidence varies with different consensus patterns.

In [ ]:
# Confidence analysis with different consensus patterns
def analyze_confidence_patterns():
    """Analyze confidence under different voting patterns."""
    
    # Define different consensus scenarios
    scenarios = [
        {"name": "Unanimous (5/5)", "votes": [5, 0, 0], "total": 5},
        {"name": "Strong Majority (4/5)", "votes": [4, 1, 0], "total": 5},
        {"name": "Simple Majority (3/5)", "votes": [3, 2, 0], "total": 5},
        {"name": "Plurality (3/5/2)", "votes": [3, 1, 1], "total": 5},
        {"name": "Close Split (3/5/2)", "votes": [3, 2, 0], "total": 5},
        {"name": "Perfect Tie (2/2/1)", "votes": [2, 2, 1], "total": 5},
    ]
    
    results = []
    
    for scenario in scenarios:
        votes = scenario["votes"]
        total = scenario["total"]
        winner_votes = max(votes)
        confidence = winner_votes / total
        
        # Determine consensus strength
        if winner_votes == total:
            strength = "Unanimous"
        elif winner_votes > total * 0.75:
            strength = "Strong Majority"
        elif winner_votes > total / 2:
            strength = "Majority"
        elif votes.count(winner_votes) == 1:
            strength = "Plurality"
        else:
            strength = "Tie"
        
        results.append({
            'scenario': scenario["name"],
            'vote_distribution': f"{votes[0]}-{votes[1]}-{votes[2]}",
            'confidence': confidence,
            'strength': strength,
            'winner_votes': winner_votes,
            'total_votes': total
        })
    
    return pd.DataFrame(results)

# Run confidence analysis
confidence_df = analyze_confidence_patterns()

# Display results
print("📊 Confidence Analysis Across Different Voting Patterns")
print("=" * 60)
display_df = confidence_df[['scenario', 'vote_distribution', 'confidence', 'strength']].copy()
display_df['confidence'] = display_df['confidence'].apply(lambda x: f"{x:.1%}")
print(display_df.to_string(index=False))

# Visualize confidence patterns
plt.figure(figsize=(12, 5))

# Confidence by scenario
plt.subplot(1, 2, 1)
bars = plt.bar(range(len(confidence_df)), confidence_df['confidence'], 
               color=plt.cm.RdYlGn(confidence_df['confidence']))
plt.xlabel('Voting Scenario')
plt.ylabel('Confidence Level')
plt.title('Confidence by Voting Pattern')
plt.xticks(range(len(confidence_df)), confidence_df['scenario'], rotation=45, ha='right')
plt.ylim(0, 1)

# Add confidence percentage labels on bars
for i, bar in enumerate(bars):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 0.02,
             f'{height:.1%}', ha='center', va='bottom')

# Consensus strength distribution
plt.subplot(1, 2, 2)
strength_counts = confidence_df['strength'].value_counts()
plt.pie(strength_counts.values, labels=strength_counts.index, autopct='%1.0f%%')
plt.title('Distribution of Consensus Strength')

plt.tight_layout()
plt.show()

# Key insights
print("\n🎯 Key Insights:")
print(f"   • Highest confidence: {confidence_df.loc[confidence_df['confidence'].idxmax(), 'scenario']} ({confidence_df['confidence'].max():.1%})")
print(f"   • Lowest confidence: {confidence_df.loc[confidence_df['confidence'].idxmin(), 'scenario']} ({confidence_df['confidence'].min():.1%})")
print(f"   • Average confidence: {confidence_df['confidence'].mean():.1%}")
print(f"   • Confidence threshold for reliability: Usually > 60% for practical use")

## 8. Interactive Experimentation

Now you can experiment with your own questions and different parameters.

In [ ]:
# Interactive experimentation function
def run_custom_experiment(question, num_responses=5, temperature=0.7):
    """Run a custom self-consistency experiment."""
    
    if not connection_ok:
        print("❌ LLM connection not available")
        return None
    
    print(f"🧪 Custom Experiment")
    print(f"📝 Question: {question}")
    print(f"🔢 Number of responses: {num_responses}")
    print(f"🌡️  Temperature: {temperature}")
    print("\n" + "="*50)
    
    # Create custom adapter with specified temperature
    custom_adapter = LiteLLMAdapter(temperature=temperature)
    
    # Configure agent
    config = AgentConfig(
        llm_interface=custom_adapter,
        target_responses=num_responses,
        prompt_template="Think step by step and provide detailed reasoning. End with 'The answer is [your answer]'."
    )
    
    # Run experiment
    agent = SelfConsistencyAgent(config, question)
    
    print(f"⏳ Generating {num_responses} responses...")
    start_time = time.time()
    
    result = agent.process_question()
    
    end_time = time.time()
    
    # Analyze results
    answers = [resp.answer for resp in agent._llm_responses]
    answer_counts = Counter(answers)
    
    print(f"\n⏱️  Execution time: {end_time - start_time:.2f}s")
    print(f"🎯 Final answer: {result.final_answer}")
    print(f"📊 Confidence: {result.confidence:.1%}")
    
    print(f"\n📋 Answer distribution:")
    for answer, count in answer_counts.most_common():
        percentage = count / num_responses * 100
        bar = "█" * int(percentage / 5)  # Visual bar
        print(f"  '{answer}': {count} votes ({percentage:.0f}%) {bar}")
    
    print(f"\n💭 Individual responses:")
    for i, response in enumerate(agent._llm_responses, 1):
        print(f"\n  Response {i}:")
        print(f"    Answer: {response.answer}")
        print(f"    Reasoning: {response.reasoning[:100]}...")
    
    return {
        'question': question,
        'final_answer': result.final_answer,
        'confidence': result.confidence,
        'num_responses': num_responses,
        'temperature': temperature,
        'execution_time': end_time - start_time,
        'answer_distribution': dict(answer_counts)
    }

# Example experiments - modify these or add your own!
example_experiments = [
    {
        "question": "A company's revenue increased from $2M to $2.5M. What was the percentage increase?",
        "num_responses": 5,
        "temperature": 0.7
    },
    {
        "question": "If you have a dataset with 1000 samples and want to split it 70-20-10 for train-validation-test, how many samples in each set?",
        "num_responses": 3,
        "temperature": 0.5
    }
]

# Run example experiments
print("🔬 Running Example Experiments...\n")

experiment_results = []
for i, exp in enumerate(example_experiments, 1):
    print(f"\n🧪 Experiment {i}/{len(example_experiments)}")
    result = run_custom_experiment(**exp)
    if result:
        experiment_results.append(result)
    print("\n" + "="*80)

## 9. Your Turn: Custom Experiments

Use the cell below to run your own experiments with different questions and parameters.

In [ ]:
# 🚀 YOUR CUSTOM EXPERIMENT
# Modify the parameters below to test your own questions

# Example: Math problem
my_result = run_custom_experiment(
    question="A data scientist has a dataset with 500,000 rows and 20 columns. If each cell contains 8 bytes of data, how much memory (in MB) does the dataset require?",
    num_responses=7,  # Try different numbers: 3, 5, 7, 10
    temperature=0.8   # Try different values: 0.1 (deterministic) to 1.5 (creative)
)

# You can run multiple experiments by copying this block
# Try different types of questions:
# - Math problems
# - Data science scenarios  
# - Logic puzzles
# - Unit conversions
# - Statistical problems

## 10. Summary and Key Takeaways

Let's summarize what we've learned about self-consistency Chain-of-Thought reasoning.

In [ ]:
# Summary analysis
print("📚 Self-Consistency Chain-of-Thought: Key Takeaways")
print("=" * 60)

print("\n🧠 Conceptual Understanding:")
print("   • Self-consistency improves LLM accuracy through multiple sampling")
print("   • Mathematical foundation: argmax_a Σ_{i=1}^m 𝟙_a(a_i = a)")
print("   • Works by leveraging the wisdom of crowds principle")
print("   • Most effective when individual responses have >50% accuracy")

print("\n⚡ Algorithmic Insights:")
print("   • O(m) complexity using Python's Counter (not O(m²))")
print("   • Efficient hash-based counting vs nested loops")
print("   • Scales linearly with number of responses")
print("   • Practical for real-time applications")

print("\n📊 Performance Patterns:")
print("   • Higher confidence correlates with answer consistency")
print("   • Unanimous consensus (100% confidence) is ideal")
print("   • >60% confidence generally indicates reliable answers")
print("   • Ties and low confidence suggest problem ambiguity")

print("\n🛠️ Practical Applications:")
print("   • Mathematical problem solving")
print("   • Data science calculations")
print("   • Reasoning tasks requiring accuracy")
print("   • Decision support systems")

print("\n⚙️ Implementation Best Practices:")
print("   • Start with 3-5 responses for speed/accuracy balance")
print("   • Use temperature 0.7-0.8 for good reasoning diversity")
print("   • Monitor confidence levels for quality control")
print("   • Consider cost vs accuracy tradeoffs")

print("\n🔮 Future Enhancements:")
print("   • Weighted voting based on response quality")
print("   • Adaptive response count based on confidence")
print("   • Integration with other reasoning techniques")
print("   • Confidence-based answer filtering")

if connection_ok and 'comparison_df' in locals():
    print("\n📈 Your Session Results:")
    if len(experiment_results) > 0:
        avg_confidence = np.mean([r['confidence'] for r in experiment_results])
        avg_time = np.mean([r['execution_time'] for r in experiment_results])
        print(f"   • Experiments run: {len(experiment_results)}")
        print(f"   • Average confidence: {avg_confidence:.1%}")
        print(f"   • Average execution time: {avg_time:.2f}s")

print("\n🎯 Next Steps:")
print("   1. Experiment with different question types")
print("   2. Try varying the number of responses (3, 5, 7, 10)")
print("   3. Test different temperature settings")
print("   4. Compare with single-response approaches")
print("   5. Implement in your own data science projects!")

print("\n✨ Congratulations! You've successfully explored self-consistency CoT reasoning!")

## 🔧 Troubleshooting

If you encounter issues:

### Connection Problems
```bash
# Check LiteLLM status
make litellm-status

# Start LiteLLM if not running
make litellm-install

# Test connection
make litellm-test
```

### Environment Issues
```bash
# Create .env file
make setup-env

# Check environment
make check-env
```

### Import Errors
```bash
# Install dependencies
make install

# Full setup
make setup-all
```

---

**Happy experimenting with self-consistency Chain-of-Thought reasoning! 🚀**